1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.style.use("seaborn")
pd.set_option("display.max_columns", None)

2. Load Dataset

In [ ]:
df = pd.read_csv("kc_house_data.csv")  # adjust path if needed
df.head()

3. Basic Cleaning & Feature Engineering

In [ ]:
df = df.dropna()
df = df[df["price"] > 0]

# Basement indicator
df["has_basement"] = (df["sqft_basement"] > 0).astype(int)

df.describe()

4. Mafia Task: Affordable Homes With Basements
 Requirements:
Must have a basement

Must be affordable (bottom 40% of basement-home prices)

Identify clusters of 4–5 properties

Visualize on maps & cluster plots

4.1 Filter Homes With Basements

In [ ]:
basement_homes = df[df["has_basement"] == 1]

price_threshold = basement_homes["price"].quantile(0.40)
affordable = basement_homes[basement_homes["price"] <= price_threshold]

affordable.shape

4.2 KMeans Clustering

In [ ]:
coords = affordable[["lat", "long"]]

kmeans = KMeans(n_clusters=4, random_state=42)
affordable["cluster"] = kmeans.fit_predict(coords)

affordable.head()

4.3 Cluster Plot

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=affordable,
    x="long", y="lat",
    hue="cluster",
    palette="tab10",
    s=40
)
plt.title("Clusters of Affordable Homes with Basements")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

4.4 Price Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    affordable["long"], affordable["lat"],
    c=affordable["price"], cmap="viridis", s=40
)
plt.colorbar(label="Price")
plt.title("Price Heatmap of Affordable Basement Homes")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

5. Linear Regression – Predicting House Prices

5.1 Select Features & Train Model

In [ ]:
features = ["sqft_living", "bedrooms", "bathrooms", "floors", "sqft_lot", "lat", "long"]
X = df[features]
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

linreg = LinearRegression()
linreg.fit(X_train, y_train)

y_pred = linreg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mse

5.2 True vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.xlabel("True Prices")
plt.ylabel("Predicted Prices")
plt.title("Linear Regression: True vs Predicted Prices")
plt.show()

6. Logistic Regression – Classifying Expensive Houses

6.1 Create Binary Target

In [ ]:
median_price = df["price"].median()
df["expensive"] = (df["price"] > median_price).astype(int)

X = df[features]
y = df["expensive"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

logreg = LogisticRegression(max_iter=200)
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
accuracy